# ClinicalTrials.gov Demographics Extraction

This notebook extracts ~1,000 clinical trials with balanced sampling across years (2009-2026).

**Runtime:** Approximately 50 minutes for 1,000 studies

**Cost:** FREE (ClinicalTrials.gov API is completely free)

## Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone your repository
!git clone https://github.com/michaeldgreenphd/clinical-trial-populations.git
%cd clinical-trial-populations

# Checkout the branch with the UPDATED extraction scripts (includes new fields)
!git checkout claude/fix-subgroup-display-BQKIP

In [ ]:
# Install dependencies
!pip install -q requests pyyaml rapidfuzz tqdm

# Verify extraction scripts are available
import os
print("✅ Checking extraction scripts...")
if os.path.exists('src/extract_all.py'):
    print("  ✓ extract_all.py found")
else:
    print("  ✗ extract_all.py NOT FOUND")
    
if os.path.exists('src/extract_stratified.py'):
    print("  ✓ extract_stratified.py found")
else:
    print("  ✗ extract_stratified.py NOT FOUND")
    print("  → Will use extract_all.py instead")
    
print("\n✅ Ready to extract data!")

## Step 2: Test with 10 Studies (Quick Test)

Let's first test with 10 studies to make sure everything works (~30 seconds)

**Note:** We'll use the basic extraction script for the test, then stratified sampling for the full extraction.

In [ ]:
# Quick test with 10 studies using basic extraction
!PYTHONPATH=. python src/extract_all.py --output data/test_10.json --limit 10

In [ ]:
# Verify the test data
import json

with open('data/test_10.json') as f:
    test_data = json.load(f)
    print(f"✅ Test successful! Extracted {len(test_data['data'])} studies")
    print(f"\nSample study: {test_data['data'][0]['nct_id']} - {test_data['data'][0]['brief_title'][:60]}...")

## Step 3: Extract 1,000 Studies (Takes ~50 minutes)

⏱️ **This will take approximately 50 minutes.** You can minimize this tab and come back later.

**Two extraction options:**
- **Option A (Recommended):** Stratified sampling - balanced ~100 studies per year
- **Option B (Fallback):** Basic extraction - first 1,000 studies with results

Run ONE of the cells below (try Option A first, use Option B if it fails):

In [ ]:
# OPTION A: Stratified sampling (RECOMMENDED - balanced by year)
# Run this cell first - if it fails, use Option B below

import os
if os.path.exists('src/extract_stratified.py'):
    !PYTHONPATH=. python src/extract_stratified.py --output data/demographics.json --total 1000 --seed 42
else:
    print("⚠️  extract_stratified.py not found. Please run Option B below instead.")

In [ ]:
# OPTION B: Basic extraction (FALLBACK - if Option A failed)
# Only run this if Option A didn't work

!PYTHONPATH=. python src/extract_all.py --output data/demographics.json --limit 1000

# Load and verify the extracted data
import json
from collections import Counter

try:
    with open('data/demographics.json') as f:
        data = json.load(f)
    
    studies = data.get('data', [])
    
    if not studies:
        print("⚠️  No studies found in the data file!")
        print("The extraction may have failed. Check the extraction output above.")
    else:
        print(f"📊 Total studies extracted: {len(studies)}")
        print(f"📅 Extraction date: {data.get('extracted_at', 'Unknown')}")
        
        # Distribution by year
        years = [s.get('results_date', '')[:4] for s in studies if s.get('results_date')]
        year_counts = Counter(years)
        print(f"\n📈 Distribution by year:")
        for year in sorted(year_counts.keys()):
            print(f"  {year}: {year_counts[year]} studies")
        
        # Reporting rates
        race_reporting = sum(1 for s in studies if s.get('race', {}).get('reported'))
        ethnicity_reporting = sum(1 for s in studies if s.get('ethnicity', {}).get('reported'))
        sex_reporting = sum(1 for s in studies if s.get('sex', {}).get('reported'))
        
        print(f"\n✅ Reporting rates:")
        if len(studies) > 0:
            print(f"  Race: {race_reporting} ({race_reporting/len(studies)*100:.1f}%)")
            print(f"  Ethnicity: {ethnicity_reporting} ({ethnicity_reporting/len(studies)*100:.1f}%)")
            print(f"  Sex: {sex_reporting} ({sex_reporting/len(studies)*100:.1f}%)")
        
        # Sample studies
        print(f"\n📋 Sample studies:")
        for i, study in enumerate(studies[:3]):
            nct_id = study.get('nct_id', 'Unknown')
            title = study.get('brief_title', 'No title')[:50]
            enrollment = study.get('enrollment', 0)
            countries = study.get('countries', [])
            
            # Format countries
            if countries:
                country_str = ', '.join(countries[:2])
                if len(countries) > 2:
                    country_str += f" +{len(countries)-2} more"
            else:
                country_str = 'N/A'
            
            print(f"  {i+1}. {nct_id}: {title}...")
            print(f"     Enrollment: {enrollment:,}, Countries: {country_str}")
            print(f"     Link: https://clinicaltrials.gov/study/{nct_id}")

except FileNotFoundError:
    print("❌ Error: data/demographics.json not found!")
    print("Make sure the extraction in Step 3 completed successfully.")
except json.JSONDecodeError as e:
    print(f"❌ Error: Invalid JSON in data/demographics.json")
    print(f"Details: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Load and verify the extracted data
import json
from collections import Counter

with open('data/demographics.json') as f:
    data = json.load(f)

studies = data['data']
print(f"📊 Total studies extracted: {len(studies)}")
print(f"📅 Extraction date: {data['extracted_at']}")

# Distribution by year
years = [s['results_date'][:4] for s in studies if s.get('results_date')]
year_counts = Counter(years)
print(f"\n📈 Distribution by year:")
for year in sorted(year_counts.keys()):
    print(f"  {year}: {year_counts[year]} studies")

# Reporting rates
race_reporting = sum(1 for s in studies if s.get('race', {}).get('reported'))
ethnicity_reporting = sum(1 for s in studies if s.get('ethnicity', {}).get('reported'))
sex_reporting = sum(1 for s in studies if s.get('sex', {}).get('reported'))

print(f"\n✅ Reporting rates:")
print(f"  Race: {race_reporting} ({race_reporting/len(studies)*100:.1f}%)")
print(f"  Ethnicity: {ethnicity_reporting} ({ethnicity_reporting/len(studies)*100:.1f}%)")
print(f"  Sex: {sex_reporting} ({sex_reporting/len(studies)*100:.1f}%)")

# Check for enhanced fields
has_breakdowns = sum(1 for s in studies if s.get('raceBreakdown'))
has_design_fields = sum(1 for s in studies if s.get('intervention_model') and s.get('intervention_model') != 'N/A')

print(f"\n🎯 Enhanced fields:")
print(f"  Studies with demographic breakdowns: {has_breakdowns} ({has_breakdowns/len(studies)*100:.1f}%)")
print(f"  Studies with design fields: {has_design_fields} ({has_design_fields/len(studies)*100:.1f}%)")

# Sample studies
print(f"\n📋 Sample studies:")
for i, study in enumerate(studies[:3]):
    # Handle countries that might be dict or string
    countries_list = study.get('countries', [])
    if countries_list:
        if isinstance(countries_list[0], dict):
            country_names = [c.get('country', 'Unknown') for c in countries_list[:2]]
        else:
            country_names = countries_list[:2]
        countries = ', '.join(country_names)
        if len(countries_list) > 2:
            countries += f" +{len(countries_list)-2} more"
    else:
        countries = 'N/A'
    
    print(f"  {i+1}. {study['nct_id']}: {study['brief_title'][:50]}...")
    print(f"     Enrollment: {study['enrollment']}, Countries: {countries}")
    print(f"     Design: {study.get('intervention_model', 'N/A')}, Sponsor: {study.get('lead_sponsor_name', 'N/A')[:30]}")
    if study.get('raceBreakdown'):
        print(f"     ✓ Has race breakdown ({len(study['raceBreakdown'])} categories)")

## Step 5: Configure Git (One-time setup)

In [ ]:
# Configure Git with your information
!git config --global user.email "your-email@example.com"  # Replace with your email
!git config --global user.name "Your Name"  # Replace with your name

## Step 6: Push Data to GitHub

You'll need to authenticate with GitHub. When prompted, use a **Personal Access Token** (not your password).

### Create a Personal Access Token:
1. Go to https://github.com/settings/tokens
2. Click "Generate new token" → "Generate new token (classic)"
3. Give it a name (e.g., "Colab Data Upload")
4. Select scopes: **repo** (full control of private repositories)
5. Click "Generate token" and **copy the token** (you won't see it again!)
6. Use this token as the password when prompted below

In [ ]:
# Data is already in the correct branch (claude/fix-subgroup-display-BQKIP)
# Just add, commit, and push

# Add the data file
!git add data/demographics.json

# Commit
!git commit -m "Update demographics data with enhanced fields (study design, endpoints, sponsor, interactive breakdowns)"

# Push (you'll be prompted for username and token)
print("\n⚠️  When prompted:")
print("   Username: michaeldgreenphd")
print("   Password: [paste your Personal Access Token]\n")

!git push origin claude/fix-subgroup-display-BQKIP

## Step 7: Download Data (Optional Backup)

In [ ]:
# Download the data file as a backup
from google.colab import files
files.download('data/demographics.json')

## ✅ All Done!

### Next Steps:
1. Wait 1-2 minutes for GitHub Pages to rebuild
2. Go to your dashboard: https://michaeldgreenphd.github.io/clinical-trial-populations/
3. Refresh the page
4. You should now see:
   - ~1,000 real clinical trials
   - Real participant counts
   - Real countries
   - **Working NCT ID links** to ClinicalTrials.gov

### Troubleshooting:
- If links still don't work, clear your browser cache and refresh
- If data doesn't appear, check that the push succeeded in Step 6
- If you see errors, check the GitHub Actions tab in your repository